# 实验二：虫害控制问题 - 方程模型

## 实验题目

在农田生态系统中，害虫按 Logistic 规律增长，益虫作为捕食者控制害虫数量。农民按固定间隔 $n$ 天喷洒杀虫剂，杀虫剂对害虫和益虫的杀灭比例分别为 $d_p$ 和 $d_n$。要求建立方程模型，判断什么条件下应使用杀虫剂，分析使用杀虫剂后系统长期状态如何变化，并结合统计数据检验实际杀虫剂使用是否合理。


## 问题背景

虫害控制的关键不只是短期降低害虫数量，还要保护益虫的自然捕食能力。如果杀虫剂对益虫的杀伤接近甚至超过对害虫的杀伤，短期内害虫可能下降，但长期可能因益虫种群被破坏而反弹。

本实验特别注意一个建模细节：无杀虫剂系统可以讨论普通微分方程平衡点；周期喷洒杀虫剂后，系统存在喷洒瞬间的跳变，一般不再收敛到固定常数平衡点，而是收敛到一个“周期平衡状态”，即每个喷洒周期重复出现的喷洒前状态和喷洒后状态。


## 数学模型建立

设：

- $P(t)$：害虫数量
- $N(t)$：益虫数量

无杀虫剂时，采用害虫-益虫耦合 Logistic 模型：

$$\frac{dP}{dt}=r_pP\left(1-\frac{P}{K_p}\right)-aPN$$

$$\frac{dN}{dt}=r_nN\left(1-\frac{N}{K_n+bP}\right)$$

其中 $K_n+bP$ 表示益虫的有效环境容量随害虫数量增加而增加。

周期喷洒杀虫剂时，若每隔 $n$ 天喷洒一次，则喷洒瞬间满足：

$$P(t_s^+)=(1-d_p)P(t_s^-)$$

$$N(t_s^+)=(1-d_n)N(t_s^-)$$

其中 $t_s^-$ 表示喷洒前，$t_s^+$ 表示喷洒后。


## 算法设计思路

1. 用 `solve_ivp` 模拟无杀虫剂连续系统，并求解无杀虫剂平衡点。
2. 用“分段积分 + 喷洒日跳变”模拟周期喷洒系统。
3. 用 Poincare 映射求周期喷洒系统的长期周期平衡：从某次喷洒后的状态出发，积分 $n$ 天到下一次喷洒前，再乘以剩余比例。若喷洒后的状态每个周期重复，则达到周期平衡。
4. 对 $d_p,d_n$ 做网格搜索，并加入喷洒间隔 $n$ 的敏感性分析。
5. 用统计数据估计实际 $d_p,d_n$，判断该杀虫剂和当前喷洒间隔是否合理。


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp, trapezoid
from scipy.optimize import root

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

OUTPUT_DIR = Path.cwd().parent / 'outputs' if Path.cwd().name == 'notebooks' else Path('math_modeling_experiments/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 参数集中定义

题目数据只有 10 天，且包含喷洒跳变，不足以稳定拟合所有微分方程参数。因此下面参数作为课堂建模假设参数，用于分析变化规律。第 0 天数据作为初始状态。


In [ ]:
params = {
    'r_p': 0.32,    # 害虫自然增长率，假设参数
    'K_p': 120.0,   # 害虫环境容量，假设参数
    'a': 0.0035,    # 益虫捕食抑制系数，假设参数
    'r_n': 0.28,    # 益虫增长率，假设参数
    'K_n': 35.0,    # 益虫基础环境容量，假设参数
    'b': 1.6,       # 害虫对益虫有效环境容量的贡献，假设参数
}

initial_state = np.array([21.0, 52.0])  # [害虫 P, 益虫 N]
simulation_days = 150
transient_start = 100
spray_interval = 3  # 题目统计数据中喷洒日为 3、6、9，因此默认 n=3

lambda_loss = 0.8
min_pest_reduction = 0.20
max_beneficial_loss = 0.30
min_selectivity_gap = 0.10

pd.DataFrame([params | {
    '初始害虫': initial_state[0],
    '初始益虫': initial_state[1],
    '默认喷洒间隔 n': spray_interval,
    '最低害虫下降比例': min_pest_reduction,
    '最高益虫损失比例': max_beneficial_loss,
    '最低选择性差值 dp-dn': min_selectivity_gap,
}])


## Python 程序实现

### 1. 定义耦合 Logistic 方程


In [ ]:
def pest_beneficial_rhs(t, y, p=params):
    P, N = np.maximum(y, 0)
    effective_capacity_n = p['K_n'] + p['b'] * P
    dP = p['r_p'] * P * (1 - P / p['K_p']) - p['a'] * P * N
    dN = p['r_n'] * N * (1 - N / effective_capacity_n)
    return [dP, dN]


def simulate_without_pesticide(days=simulation_days, y0=initial_state):
    t_eval = np.linspace(0, days, days * 2 + 1)
    sol = solve_ivp(pest_beneficial_rhs, (0, days), y0, t_eval=t_eval, rtol=1e-8, atol=1e-10)
    if not sol.success:
        raise RuntimeError(sol.message)
    return pd.DataFrame({
        '时间': sol.t,
        '害虫数量': np.maximum(sol.y[0], 0),
        '益虫数量': np.maximum(sol.y[1], 0),
    })


no_spray_result = simulate_without_pesticide()
no_spray_result.head()


### 2. 无杀虫剂平衡点

正平衡点满足 $dP/dt=0,dN/dt=0$。当 $P>0,N>0$ 时，有：

$$N^*=K_n+bP^*$$

$$r_p\left(1-\frac{P^*}{K_p}\right)-aN^*=0$$

也可以直接用 `root` 数值求解。


In [ ]:
def equilibrium_equations(x):
    return pest_beneficial_rhs(0, x)


eq_result = root(equilibrium_equations, x0=[25, 75])
if not eq_result.success:
    raise RuntimeError(eq_result.message)

equilibrium = np.maximum(eq_result.x, 0)
P_star, N_star = equilibrium
equilibrium_df = pd.DataFrame({
    '状态': ['无杀虫剂平衡点'],
    '害虫 P': [P_star],
    '益虫 N': [N_star],
}).round(3)
equilibrium_df


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(no_spray_result['时间'], no_spray_result['害虫数量'], label='害虫 P(t)', linewidth=2)
ax.plot(no_spray_result['时间'], no_spray_result['益虫数量'], label='益虫 N(t)', linewidth=2)
ax.axhline(P_star, color='C0', linestyle='--', alpha=0.7, label=f'害虫平衡值 {P_star:.2f}')
ax.axhline(N_star, color='C1', linestyle='--', alpha=0.7, label=f'益虫平衡值 {N_star:.2f}')
ax.set_title('无杀虫剂时害虫与益虫数量变化')
ax.set_xlabel('时间（日）')
ax.set_ylabel('数量（只）')
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
fig_path = OUTPUT_DIR / '02_no_pesticide_simulation.png'
plt.savefig(fig_path, dpi=150)
plt.show()
print('图表已保存：', fig_path)


无杀虫剂时，系统趋向正平衡点。根据本组假设参数，平衡点约为 $P^*=23.89,N^*=73.23$。这表示益虫长期存在时，害虫并不会无限增长，而是在自然增长、资源上限和捕食抑制共同作用下稳定在较低水平。


## 周期喷洒杀虫剂模型

周期喷洒后，系统每隔 $n$ 天发生一次跳变。此时不能简单说系统收敛到一个固定点，而应该求周期平衡状态：

$$Y_{k+1}^+=S\Phi_n(Y_k^+)$$

其中 $Y_k^+=(P_k^+,N_k^+)$ 是第 $k$ 次喷洒后的状态，$\Phi_n$ 表示连续积分 $n$ 天，$S=diag(1-d_p,1-d_n)$ 表示喷洒杀灭。周期平衡满足：

$$Y^+=S\Phi_n(Y^+)$$

这就是 Poincare 映射的不动点。


In [ ]:
def one_cycle_map(post_spray_state, n, dp, dn):
    """从某次喷洒后状态出发，积分 n 天，再执行下一次喷洒。"""
    state = np.maximum(np.asarray(post_spray_state, dtype=float), 1e-9)
    sol = solve_ivp(pest_beneficial_rhs, (0, n), state, t_eval=[n], rtol=1e-8, atol=1e-10)
    if not sol.success:
        raise RuntimeError(sol.message)
    pre_spray = np.maximum(sol.y[:, -1], 0)
    post_next = np.array([(1 - dp) * pre_spray[0], (1 - dn) * pre_spray[1]])
    return post_next, pre_spray


def simulate_periodic_pesticide(dp, dn, n=spray_interval, days=simulation_days, y0=initial_state):
    times = [0.0]
    pests = [float(y0[0])]
    beneficials = [float(y0[1])]
    state = np.array(y0, dtype=float)
    start = 0.0
    spray_records = []

    for spray_day in range(n, days + 1, n):
        t_eval = np.linspace(start, spray_day, int((spray_day - start) * 2) + 1)
        sol = solve_ivp(pest_beneficial_rhs, (start, spray_day), state, t_eval=t_eval, rtol=1e-7, atol=1e-9)
        if not sol.success:
            raise RuntimeError(sol.message)
        for t, P, N in zip(sol.t[1:], sol.y[0][1:], sol.y[1][1:]):
            times.append(float(t))
            pests.append(max(float(P), 0))
            beneficials.append(max(float(N), 0))

        pre = np.array([pests[-1], beneficials[-1]])
        post = np.array([(1 - dp) * pre[0], (1 - dn) * pre[1]])
        spray_records.append({'喷洒日': spray_day, '喷洒前害虫': pre[0], '喷洒后害虫': post[0],
                              '喷洒前益虫': pre[1], '喷洒后益虫': post[1]})
        times.append(float(spray_day))
        pests.append(post[0])
        beneficials.append(post[1])
        state = post
        start = float(spray_day)

    return pd.DataFrame({'时间': times, '害虫数量': pests, '益虫数量': beneficials}), pd.DataFrame(spray_records)


def periodic_equilibrium(dp, dn, n=spray_interval):
    """求周期喷洒系统的长期周期平衡：喷洒后点、喷洒前点和周期平均值。"""
    guess = np.array([(1 - dp) * P_star, (1 - dn) * N_star])
    for _ in range(80):
        guess, _ = one_cycle_map(guess, n, dp, dn)

    sol_root = root(lambda z: one_cycle_map(z, n, dp, dn)[0] - z, guess)
    if not sol_root.success:
        raise RuntimeError(sol_root.message)
    post = np.maximum(sol_root.x, 0)
    _, pre = one_cycle_map(post, n, dp, dn)

    t_eval = np.linspace(0, n, n * 20 + 1)
    sol = solve_ivp(pest_beneficial_rhs, (0, n), post, t_eval=t_eval, rtol=1e-8, atol=1e-10)
    avg_pest = trapezoid(sol.y[0], sol.t) / n
    avg_beneficial = trapezoid(sol.y[1], sol.t) / n
    return {
        '喷洒后害虫': post[0], '喷洒后益虫': post[1],
        '下次喷洒前害虫': pre[0], '下次喷洒前益虫': pre[1],
        '周期平均害虫': avg_pest, '周期平均益虫': avg_beneficial,
    }


下面先给出一个选择性较好的示例：$d_p=0.40,d_n=0.10,n=7$。它不是题目实测药剂，只用于说明合理药剂的长期周期平衡如何改变。


In [ ]:
example_dp, example_dn, example_n = 0.40, 0.10, 7
example_result, _ = simulate_periodic_pesticide(example_dp, example_dn, example_n)
example_cycle = periodic_equilibrium(example_dp, example_dn, example_n)

cycle_table = pd.DataFrame([
    {'状态': '无杀虫剂平衡点', '害虫 P': P_star, '益虫 N': N_star},
    {'状态': '周期平衡：喷洒后', '害虫 P': example_cycle['喷洒后害虫'], '益虫 N': example_cycle['喷洒后益虫']},
    {'状态': '周期平衡：下次喷洒前', '害虫 P': example_cycle['下次喷洒前害虫'], '益虫 N': example_cycle['下次喷洒前益虫']},
    {'状态': '周期平衡：周期平均', '害虫 P': example_cycle['周期平均害虫'], '益虫 N': example_cycle['周期平均益虫']},
]).round(3)
cycle_table


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(example_result['时间'], example_result['害虫数量'], label='害虫 P(t)', linewidth=2)
ax.plot(example_result['时间'], example_result['益虫数量'], label='益虫 N(t)', linewidth=2)
for day in range(example_n, simulation_days + 1, example_n):
    ax.axvline(day, color='gray', alpha=0.10, linewidth=1)
ax.axhline(P_star, color='C0', linestyle='--', alpha=0.55, label='无药害虫平衡')
ax.axhline(N_star, color='C1', linestyle='--', alpha=0.55, label='无药益虫平衡')
ax.set_title(f'有杀虫剂模拟：dp={example_dp}, dn={example_dn}, n={example_n}')
ax.set_xlabel('时间（日）')
ax.set_ylabel('数量（只）')
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
fig_path = OUTPUT_DIR / '02_periodic_pesticide_simulation.png'
plt.savefig(fig_path, dpi=150)
plt.show()
print('图表已保存：', fig_path)


由表格可见，使用杀虫剂后“平衡点”的改变并不是从一个固定点变成另一个固定点，而是变成周期波动：喷洒后数量较低，随后在一个周期内增长，到下一次喷洒前达到较高值。合理的杀虫剂应使周期平均害虫低于无药平衡，同时益虫周期平均不能下降过多。


## dp、dn 满足什么条件时应使用杀虫剂

这里采用三条判定标准：

- 控制有效：周期长期平均害虫数量相对无药平衡至少下降 20%；
- 生态可接受：周期长期平均益虫损失不超过 30%；
- 药剂有选择性：$d_p-d_n\ge 0.10$，即对害虫的杀灭比例应明显高于对益虫的杀灭比例。

综合指标用于排序：

$$J=\overline{P}+\lambda\max(0,N^*-\overline{N})$$

其中 $\overline{P},\overline{N}$ 为长期平均害虫和益虫数量，$J$ 越小越好。但是否建议使用仍以上述三条约束为准。


In [ ]:
def long_term_means(result, start_day=transient_start):
    tail = result[result['时间'] >= start_day]
    return tail['害虫数量'].mean(), tail['益虫数量'].mean()


def evaluate_control(dp, dn, n=spray_interval):
    result, _ = simulate_periodic_pesticide(dp, dn, n=n)
    avg_pest, avg_beneficial = long_term_means(result)
    pest_reduction = 1 - avg_pest / P_star
    beneficial_loss = max(0, 1 - avg_beneficial / N_star)
    J = avg_pest + lambda_loss * max(0, N_star - avg_beneficial)
    recommend = (
        pest_reduction >= min_pest_reduction
        and beneficial_loss <= max_beneficial_loss
        and dp - dn >= min_selectivity_gap
    )
    return avg_pest, avg_beneficial, pest_reduction, beneficial_loss, J, recommend


dp_values = np.round(np.linspace(0.05, 0.75, 25), 3)
dn_values = np.round(np.linspace(0.05, 0.75, 25), 3)
grid_records = []

for dp in dp_values:
    for dn in dn_values:
        avg_p, avg_n, red, loss, J, recommend = evaluate_control(float(dp), float(dn), n=spray_interval)
        grid_records.append({
            'dp': dp, 'dn': dn,
            '长期平均害虫': avg_p,
            '长期平均益虫': avg_n,
            '害虫下降比例': red,
            '益虫损失比例': loss,
            '综合指标J': J,
            '建议使用': recommend,
        })

grid_df = pd.DataFrame(grid_records)
grid_df.head()


In [ ]:
heatmap_data = grid_df.pivot(index='dn', columns='dp', values='综合指标J')
recommend_data = grid_df.pivot(index='dn', columns='dp', values='建议使用').astype(int)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(
    heatmap_data.values,
    origin='lower',
    aspect='auto',
    extent=[dp_values.min(), dp_values.max(), dn_values.min(), dn_values.max()],
    cmap='YlGnBu_r',
)
if recommend_data.values.max() > 0:
    contour = ax.contour(dp_values, dn_values, recommend_data.values, levels=[0.5], colors='red', linewidths=2)
    ax.clabel(contour, inline=True, fmt={0.5: '建议使用边界'}, fontsize=9)
ax.plot([dp_values.min(), dp_values.max()], [dp_values.min(), dp_values.max()], 'k--', alpha=0.45, label='dn = dp')
ax.set_title(f'dp-dn 控制效果热力图（固定 n={spray_interval}）')
ax.set_xlabel('dp：对害虫杀灭比例')
ax.set_ylabel('dn：对益虫杀灭比例')
ax.legend(loc='upper right')
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('综合指标 J，越小越好')
plt.tight_layout()
fig_path = OUTPUT_DIR / '02_dp_dn_effect_heatmap.png'
plt.savefig(fig_path, dpi=150)
plt.show()
print('图表已保存：', fig_path)


In [ ]:
recommended_df = grid_df[grid_df['建议使用']].copy()
if recommended_df.empty:
    condition_summary = pd.DataFrame([{'结论': f'固定 n={spray_interval} 时，在当前网格和判定标准下没有建议使用区域'}])
else:
    condition_summary = pd.DataFrame({
        '指标': ['建议 dp 范围', '建议 dn 范围', '建议网格点数', '解释'],
        '结果': [
            f"{recommended_df['dp'].min():.3f} - {recommended_df['dp'].max():.3f}",
            f"{recommended_df['dn'].min():.3f} - {recommended_df['dn'].max():.3f}",
            len(recommended_df),
            '该范围依赖 n=3 和本实验判定标准；若 n 改变，范围也会改变。',
        ]
    })
display(condition_summary)

best_recommended = recommended_df.sort_values('综合指标J').head(5)
best_recommended[['dp', 'dn', '长期平均害虫', '长期平均益虫', '害虫下降比例', '益虫损失比例', '综合指标J']].round(3)


从热力图和建议区域可以看到：在固定 $n=3$ 的高频喷洒条件下，只有 $d_p$ 明显大于 $d_n$ 且益虫损失不大的组合才建议使用。若 $d_n$ 接近或大于 $d_p$，即使短期杀死了一部分害虫，也会削弱益虫捕食控制，长期效果往往不好。


## 喷洒间隔 n 是否应该调整

下面进一步比较不同喷洒间隔。注意：如果药剂本身选择性不好，即 $d_n\ge d_p$，单纯调整 $n$ 通常不能从根本上解决问题；如果药剂选择性较好，则适当增大 $n$ 可以减少对益虫的持续伤害。


In [ ]:
n_values = [3, 5, 7, 10, 14]
coarse_dp = np.round(np.linspace(0.05, 0.75, 15), 3)
coarse_dn = np.round(np.linspace(0.05, 0.75, 15), 3)
n_summary_rows = []

for n in n_values:
    records = []
    for dp in coarse_dp:
        for dn in coarse_dn:
            avg_p, avg_n, red, loss, J, recommend = evaluate_control(float(dp), float(dn), n=n)
            records.append({'dp': dp, 'dn': dn, '平均害虫': avg_p, '平均益虫': avg_n,
                            '害虫下降比例': red, '益虫损失比例': loss, 'J': J, '建议': recommend})
    tmp = pd.DataFrame(records)
    rec = tmp[tmp['建议']]
    if rec.empty:
        n_summary_rows.append({'n': n, '建议组合数': 0, '建议 dp 范围': '无', '建议 dn 范围': '无', '最小J': np.nan})
    else:
        n_summary_rows.append({
            'n': n,
            '建议组合数': len(rec),
            '建议 dp 范围': f"{rec['dp'].min():.2f} - {rec['dp'].max():.2f}",
            '建议 dn 范围': f"{rec['dn'].min():.2f} - {rec['dn'].max():.2f}",
            '最小J': rec['J'].min(),
        })

n_summary = pd.DataFrame(n_summary_rows)
n_summary


由 $n$ 的敏感性分析可见：喷洒越频繁，对益虫的持续压力越大；喷洒间隔增大后，若药剂有足够选择性，允许的 $d_p$ 范围可以变宽。但如果药剂本身对益虫杀伤更强，调整 $n$ 只能减轻破坏，不能使其成为真正合理的长期防治方案。


## 参考统计数据进行模型检验

题目给出的数据如下，喷洒日为 3、6、9 日。由于数据是按日统计，不能精确得到喷洒瞬间的 $t^-$ 和 $t^+$，因此用喷洒日前一天到喷洒日的下降比例近似估计实际 $d_p,d_n$。


In [ ]:
observed_df = pd.DataFrame({
    '时间(日)': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
    '益虫数量(只)': [52, 68, 89, 45, 62, 81, 38, 55, 74, 32],
    '害虫数量(只)': [21, 24, 28, 18, 22, 26, 14, 19, 23, 12],
    '杀虫剂': [0, 0, 0, 1, 0, 0, 1, 0, 0, 1],
})
observed_df


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(observed_df['时间(日)'], observed_df['害虫数量(只)'], marker='o', label='实测害虫数量')
ax.plot(observed_df['时间(日)'], observed_df['益虫数量(只)'], marker='o', label='实测益虫数量')
for day in observed_df.loc[observed_df['杀虫剂'] == 1, '时间(日)']:
    ax.axvline(day, color='red', linestyle='--', alpha=0.55)
    ax.text(day + 0.05, ax.get_ylim()[1] * 0.90, '喷洒', color='red')
ax.set_title('实测数据中害虫与益虫数量变化')
ax.set_xlabel('时间（日）')
ax.set_ylabel('数量（只）')
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
fig_path = OUTPUT_DIR / '02_observed_data_curve.png'
plt.savefig(fig_path, dpi=150)
plt.show()
print('图表已保存：', fig_path)


In [ ]:
kill_estimates = []
for day in observed_df.loc[observed_df['杀虫剂'] == 1, '时间(日)']:
    prev = observed_df.loc[observed_df['时间(日)'] == day - 1].iloc[0]
    curr = observed_df.loc[observed_df['时间(日)'] == day].iloc[0]
    dp_hat = (prev['害虫数量(只)'] - curr['害虫数量(只)']) / prev['害虫数量(只)']
    dn_hat = (prev['益虫数量(只)'] - curr['益虫数量(只)']) / prev['益虫数量(只)']
    kill_estimates.append({
        '喷洒日': int(day),
        '害虫下降比例 dp估计': dp_hat,
        '益虫下降比例 dn估计': dn_hat,
        'dn 是否不小于 dp': dn_hat >= dp_hat,
    })

kill_df = pd.DataFrame(kill_estimates)
estimated_dp = kill_df['害虫下降比例 dp估计'].mean()
estimated_dn = kill_df['益虫下降比例 dn估计'].mean()
display(kill_df.style.format({'害虫下降比例 dp估计': '{:.1%}', '益虫下降比例 dn估计': '{:.1%}'}))

estimated_table = pd.DataFrame({
    '参数': ['估计 dp', '估计 dn', 'dn - dp'],
    '数值': [estimated_dp, estimated_dn, estimated_dn - estimated_dp],
}).round(3)
estimated_table


统计数据估计得到的实际杀灭比例中，$d_n$ 大于 $d_p$，说明杀虫剂对益虫的伤害反而更大。这违反了“药剂应对害虫更有选择性”的基本要求。


In [ ]:
actual_result, _ = simulate_periodic_pesticide(estimated_dp, estimated_dn, n=spray_interval)
actual_cycle = periodic_equilibrium(estimated_dp, estimated_dn, n=spray_interval)
actual_avg_p, actual_avg_n = actual_cycle['周期平均害虫'], actual_cycle['周期平均益虫']
actual_pest_reduction = 1 - actual_avg_p / P_star
actual_beneficial_loss = 1 - actual_avg_n / N_star

actual_cycle_table = pd.DataFrame([
    {'状态': '无杀虫剂平衡点', '害虫 P': P_star, '益虫 N': N_star},
    {'状态': '实际药剂周期平衡：喷洒后', '害虫 P': actual_cycle['喷洒后害虫'], '益虫 N': actual_cycle['喷洒后益虫']},
    {'状态': '实际药剂周期平衡：下次喷洒前', '害虫 P': actual_cycle['下次喷洒前害虫'], '益虫 N': actual_cycle['下次喷洒前益虫']},
    {'状态': '实际药剂周期平衡：周期平均', '害虫 P': actual_avg_p, '益虫 N': actual_avg_n},
]).round(3)

print(f'实际估计 dp={estimated_dp:.3f}, dn={estimated_dn:.3f}, n={spray_interval}')
print(f'害虫相对无药平衡下降比例：{actual_pest_reduction:.1%}')
print(f'益虫相对无药平衡损失比例：{actual_beneficial_loss:.1%}')
actual_cycle_table


In [ ]:
actual_n_rows = []
for n in [3, 5, 7, 10, 14, 21, 30]:
    cycle = periodic_equilibrium(estimated_dp, estimated_dn, n=n)
    avg_p, avg_n = cycle['周期平均害虫'], cycle['周期平均益虫']
    actual_n_rows.append({
        'n': n,
        '周期平均害虫': avg_p,
        '周期平均益虫': avg_n,
        '害虫下降比例': 1 - avg_p / P_star,
        '益虫损失比例': 1 - avg_n / N_star,
        '是否建议': (1 - avg_p / P_star >= min_pest_reduction)
                 and (1 - avg_n / N_star <= max_beneficial_loss)
                 and (estimated_dp - estimated_dn >= min_selectivity_gap),
    })

actual_n_table = pd.DataFrame(actual_n_rows)
actual_n_table.round(3)


## 一张图汇总所有关键可视化

下面用一张 dashboard 汇总本实验最关键的可视化结果：喷洒间隔 $n$ 对建议区域的影响、无杀虫剂模拟、有杀虫剂模拟、$d_p-d_n$ 控制效果热力图以及实测数据曲线。

In [ ]:
def parse_range_text(value):
    if value == '无' or pd.isna(value):
        return np.nan, np.nan
    left, right = str(value).split(' - ')
    return float(left), float(right)


n_plot = n_summary.copy()
n_plot[['dp_min', 'dp_max']] = n_plot['建议 dp 范围'].apply(lambda x: pd.Series(parse_range_text(x)))
n_plot[['dn_min', 'dn_max']] = n_plot['建议 dn 范围'].apply(lambda x: pd.Series(parse_range_text(x)))

fig = plt.figure(figsize=(16, 13))
gs = fig.add_gridspec(3, 2, height_ratios=[1.05, 1.25, 1.25], hspace=0.38, wspace=0.24)

ax_n = fig.add_subplot(gs[0, 0])
bars = ax_n.bar(n_plot['n'], n_plot['建议组合数'], color='#457b9d', width=1.15, label='建议组合数')
ax_n.set_title('喷洒间隔 n 对建议组合数和最小 J 的影响')
ax_n.set_xlabel('喷洒间隔 n（日）')
ax_n.set_ylabel('建议组合数')
ax_n.grid(axis='y', alpha=0.25)
for bar in bars:
    ax_n.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8, f'{bar.get_height():.0f}', ha='center', fontsize=9)
ax_j = ax_n.twinx()
ax_j.plot(n_plot['n'], n_plot['最小J'], marker='o', color='#e76f51', linewidth=2, label='最小 J')
ax_j.set_ylabel('最小 J')
lines1, labels1 = ax_n.get_legend_handles_labels()
lines2, labels2 = ax_j.get_legend_handles_labels()
ax_n.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

ax_range = fig.add_subplot(gs[0, 1])
ax_range.fill_between(n_plot['n'], n_plot['dp_min'], n_plot['dp_max'], color='#2a9d8f', alpha=0.24, label='建议 dp 范围')
ax_range.plot(n_plot['n'], n_plot['dp_min'], marker='o', color='#2a9d8f', linewidth=1.5)
ax_range.plot(n_plot['n'], n_plot['dp_max'], marker='o', color='#1b6f63', linewidth=1.5)
ax_range.fill_between(n_plot['n'], n_plot['dn_min'], n_plot['dn_max'], color='#f4a261', alpha=0.30, label='建议 dn 范围')
ax_range.plot(n_plot['n'], n_plot['dn_min'], marker='s', color='#f4a261', linewidth=1.5)
ax_range.plot(n_plot['n'], n_plot['dn_max'], marker='s', color='#c76f2a', linewidth=1.5)
ax_range.set_title('建议 dp、dn 范围随 n 的变化')
ax_range.set_xlabel('喷洒间隔 n（日）')
ax_range.set_ylabel('杀灭比例范围')
ax_range.set_ylim(0, 0.82)
ax_range.grid(alpha=0.25)
ax_range.legend(loc='upper left')

ax_no = fig.add_subplot(gs[1, 0])
ax_no.plot(no_spray_result['时间'], no_spray_result['害虫数量'], label='害虫 P(t)', linewidth=2)
ax_no.plot(no_spray_result['时间'], no_spray_result['益虫数量'], label='益虫 N(t)', linewidth=2)
ax_no.axhline(P_star, color='C0', linestyle='--', alpha=0.65, label=f'害虫平衡 {P_star:.2f}')
ax_no.axhline(N_star, color='C1', linestyle='--', alpha=0.65, label=f'益虫平衡 {N_star:.2f}')
ax_no.set_title('无杀虫剂模拟')
ax_no.set_xlabel('时间（日）')
ax_no.set_ylabel('数量（只）')
ax_no.grid(alpha=0.25)
ax_no.legend(fontsize=9)

ax_periodic = fig.add_subplot(gs[1, 1])
ax_periodic.plot(example_result['时间'], example_result['害虫数量'], label='害虫 P(t)', linewidth=2)
ax_periodic.plot(example_result['时间'], example_result['益虫数量'], label='益虫 N(t)', linewidth=2)
for day in range(example_n, simulation_days + 1, example_n):
    ax_periodic.axvline(day, color='gray', alpha=0.08, linewidth=1)
ax_periodic.axhline(P_star, color='C0', linestyle='--', alpha=0.45, label='无药害虫平衡')
ax_periodic.axhline(N_star, color='C1', linestyle='--', alpha=0.45, label='无药益虫平衡')
ax_periodic.set_title(f'有杀虫剂模拟：dp={example_dp}, dn={example_dn}, n={example_n}')
ax_periodic.set_xlabel('时间（日）')
ax_periodic.set_ylabel('数量（只）')
ax_periodic.grid(alpha=0.25)
ax_periodic.legend(fontsize=9)

ax_heat = fig.add_subplot(gs[2, 0])
im = ax_heat.imshow(
    heatmap_data.values,
    origin='lower',
    aspect='auto',
    extent=[dp_values.min(), dp_values.max(), dn_values.min(), dn_values.max()],
    cmap='YlGnBu_r',
)
if recommend_data.values.max() > 0:
    ax_heat.contour(dp_values, dn_values, recommend_data.values, levels=[0.5], colors='red', linewidths=2)
ax_heat.plot([dp_values.min(), dp_values.max()], [dp_values.min(), dp_values.max()], 'k--', alpha=0.45, label='dn = dp')
ax_heat.set_title(f'dp-dn 控制效果热力图（n={spray_interval}）')
ax_heat.set_xlabel('dp：害虫杀灭比例')
ax_heat.set_ylabel('dn：益虫杀灭比例')
ax_heat.legend(loc='upper right', fontsize=9)
cbar = fig.colorbar(im, ax=ax_heat, fraction=0.046, pad=0.04)
cbar.set_label('综合指标 J')

ax_obs = fig.add_subplot(gs[2, 1])
ax_obs.plot(observed_df['时间(日)'], observed_df['害虫数量(只)'], marker='o', label='实测害虫数量')
ax_obs.plot(observed_df['时间(日)'], observed_df['益虫数量(只)'], marker='o', label='实测益虫数量')
for day in observed_df.loc[observed_df['杀虫剂'] == 1, '时间(日)']:
    ax_obs.axvline(day, color='red', linestyle='--', alpha=0.50)
    ax_obs.text(day + 0.05, ax_obs.get_ylim()[1] * 0.90, '喷洒', color='red', fontsize=9)
ax_obs.set_title('实测数据曲线与喷洒日期')
ax_obs.set_xlabel('时间（日）')
ax_obs.set_ylabel('数量（只）')
ax_obs.grid(alpha=0.25)
ax_obs.legend(fontsize=9)

fig.suptitle('虫害控制方程模型关键可视化总览', fontsize=18)
dashboard_path = OUTPUT_DIR / '02_complete_visual_dashboard.png'
plt.savefig(dashboard_path, dpi=150, bbox_inches='tight')
plt.show()
print('总览图已保存：', dashboard_path)


这张表回答了“是否应该调整喷洒间隔 $n$”的问题：对题目数据估计出的实际药剂而言，$d_n>d_p$，药剂选择性本身不好。增大 $n$ 可以减轻益虫损失，但并不能让长期害虫水平显著低于无药平衡。因此，仅调整 $n$ 不能使该药剂成为合理方案；更合理的是更换为对害虫更有选择性的药剂，或采用生物防治、低频低剂量综合防治。


## 结论分析

1. 无杀虫剂时，害虫-益虫耦合 Logistic 系统趋向正平衡点。本实验参数下，平衡点约为 $P^*=23.89,N^*=73.23$。

2. 使用周期性杀虫剂后，系统不是收敛到另一个普通固定平衡点，而是变为周期平衡状态：每个周期有喷洒后低点、下次喷洒前高点和周期平均值。所谓“平衡点改变”，更准确地说是从无药固定平衡点变为有药周期平衡轨道。

3. 杀虫剂建议使用条件应同时满足：$d_p$ 明显大于 $d_n$，长期平均害虫数量显著低于无药平衡，且益虫长期损失不过大。在本 Notebook 的判定标准中，要求害虫下降至少 20%、益虫损失不超过 30%、且 $d_p-d_n\ge0.10$。

4. 题目统计数据估计得到 $d_p\approx0.432,d_n\approx0.531$。益虫下降比例大于害虫下降比例，说明该杀虫剂选择性差。模型检验显示，在 $n=3$ 下它会严重压低益虫，并使长期害虫水平反而高于无杀虫剂平衡，因此不合理。

5. 对该实际药剂而言，单纯调整喷洒间隔 $n$ 不是根本解决办法。增大 $n$ 可以降低对益虫的破坏，但由于 $d_n>d_p$，它仍不满足合理杀虫剂的选择性条件。若必须使用杀虫剂，应优先更换为 $d_p$ 较大、$d_n$ 较小的选择性药剂，并根据网格搜索结果选择合适的 $n$。

6. 模型局限性：统计数据较少，喷洒瞬间的 $t^-,t^+$ 未被直接观测，参数主要为假设值；因此本结论适合作为课堂建模判断，不应直接替代真实农田实验和专业植保决策。
